# CS3807 – Deep Learning Laboratory
## Experiment 3: Implementation of Convolutional Neural Networks (CNNs) for Image Classification

**Objective:** To understand the working principle of Convolutional Neural Networks by implementing
convolution, pooling, feature map visualization, and image classification using TensorFlow/Keras.

**Dataset:** CIFAR-10 (50,000 training images, 10,000 testing images, 10 classes, 32×32×3)


## 0. Setup — Imports and Configuration

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from sklearn.metrics import (confusion_matrix, classification_report,
                              precision_score, recall_score, f1_score, accuracy_score)
import time

# Reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))


## Task 1 — Load CIFAR-10
- Display ten sample images
- Print dataset dimensions
- Plot class distribution


In [ ]:
# Load CIFAR-10
(X_train, y_train), (X_test, y_test) = keras.datasets.cifar10.load_data()

y_train = y_train.flatten()
y_test = y_test.flatten()

class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

print("Training images shape :", X_train.shape)
print("Training labels shape :", y_train.shape)
print("Testing images shape  :", X_test.shape)
print("Testing labels shape  :", y_test.shape)
print("Number of classes     :", len(class_names))


In [ ]:
# Display 10 sample images
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
rng = np.random.default_rng(SEED)
sample_idx = rng.choice(len(X_train), 10, replace=False)

for ax, idx in zip(axes.flat, sample_idx):
    ax.imshow(X_train[idx])
    ax.set_title(class_names[y_train[idx]])
    ax.axis('off')

plt.suptitle("Sample Images from CIFAR-10")
plt.tight_layout()
plt.show()


In [ ]:
# Plot class distribution in the training set
unique, counts = np.unique(y_train, return_counts=True)

plt.figure(figsize=(9, 5))
plt.bar([class_names[i] for i in unique], counts, color='#1f77b4')
plt.title("Class Distribution in Training Set")
plt.xlabel("Class")
plt.ylabel("Count")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("Images per class:", dict(zip([class_names[i] for i in unique], counts)))


**Inference:** The dataset consists of low-resolution (32×32) colour images spanning 10 visually
distinct object classes. The small size and varying backgrounds indicate that the classification task
requires the network to learn robust, generalizable features rather than relying on fine texture detail.
The training set is perfectly balanced, with exactly 5,000 images per class, so no class-weighting or
resampling is required.

## 3.1 Common Convolution Kernels

Demonstration of classic hand-crafted kernels (identity, Sobel-X, Sobel-Y, Laplacian, sharpen, box blur,
Gaussian blur, emboss, outline, motion blur) applied to a sample grayscale image, to build intuition for
what a convolution kernel does before training a learned CNN.


In [ ]:
import cv2

kernels = {
    "Identity": np.array([[0,0,0],[0,1,0],[0,0,0]]),
    "Sobel-X": np.array([[-1,0,1],[-2,0,2],[-1,0,1]]),
    "Sobel-Y": np.array([[-1,-2,-1],[0,0,0],[1,2,1]]),
    "Laplacian": np.array([[0,-1,0],[-1,4,-1],[0,-1,0]]),
    "Sharpen": np.array([[0,-1,0],[-1,5,-1],[0,-1,0]]),
    "Box Blur": (1/9)*np.ones((3,3)),
    "Gaussian Blur": (1/16)*np.array([[1,2,1],[2,4,2],[1,2,1]]),
    "Emboss": np.array([[-2,-1,0],[-1,1,1],[0,1,2]]),
    "Outline": np.array([[-1,-1,-1],[-1,8,-1],[-1,-1,-1]]),
}

sample_gray = cv2.cvtColor(X_train[sample_idx[0]], cv2.COLOR_RGB2GRAY).astype(np.float32)

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
axes = axes.flatten()
axes[0].imshow(sample_gray, cmap='gray')
axes[0].set_title("Original")
axes[0].axis('off')

for ax, (name, k) in zip(axes[1:], kernels.items()):
    filtered = cv2.filter2D(sample_gray, -1, k)
    ax.imshow(filtered, cmap='gray')
    ax.set_title(name)
    ax.axis('off')

plt.suptitle("Effect of Common Convolution Kernels")
plt.tight_layout()
plt.show()


## Numerical Examples (verification of theory)

### Example 1 — Convolution
$$X=\begin{bmatrix}1&2&3\\4&5&6\\7&8&9\end{bmatrix}, \quad K=\begin{bmatrix}1&0\\0&1\end{bmatrix}$$


In [ ]:
X_ex = np.array([[1,2,3],[4,5,6],[7,8,9]], dtype=float)
K_ex = np.array([[1,0],[0,1]], dtype=float)

def conv2d_valid(X, K):
    kh, kw = K.shape
    oh, ow = X.shape[0]-kh+1, X.shape[1]-kw+1
    out = np.zeros((oh, ow))
    for i in range(oh):
        for j in range(ow):
            out[i, j] = np.sum(X[i:i+kh, j:j+kw] * K)
    return out

feature_map = conv2d_valid(X_ex, K_ex)
print("Feature Map:\n", feature_map)


### Example 2 — Max Pooling (2×2 window)
$$\begin{bmatrix}1&5&2&3\\7&8&1&0\\4&6&9&5\\2&3&1&8\end{bmatrix}$$


In [ ]:
def max_pool2d(X, size=2, stride=2):
    h, w = X.shape
    oh, ow = h // stride, w // stride
    out = np.zeros((oh, ow))
    for i in range(oh):
        for j in range(ow):
            window = X[i*stride:i*stride+size, j*stride:j*stride+size]
            out[i, j] = np.max(window)
    return out

feat = np.array([[1,5,2,3],[7,8,1,0],[4,6,9,5],[2,3,1,8]], dtype=float)
print("Max-Pooled Output:\n", max_pool2d(feat))


### Example 3 — Trainable Parameter Calculation
For a 32×32×3 input, Conv2D with 16 filters of size 3×3:
$$\text{Params} = (3\times3\times3 + 1)\times16 = 448$$


In [ ]:
params = (3*3*3 + 1) * 16
print("Total trainable parameters:", params)


## Task 2 — Implement a Convolution Layer
Compare kernel sizes 3×3, 5×5, 7×7 and record the resulting feature map sizes, for both `same` and
`valid` padding, on a 32×32 input with stride 1.


In [ ]:
def output_size(N, F, P, S):
    return (N - F + 2*P) // S + 1

input_size = 32

print(f"{'Kernel':<10}{'Padding':<10}{'Stride':<8}{'Output Size'}")
for k in [3, 5, 7]:
    # 'same' padding output equals input size for stride 1
    print(f"{k}x{k:<7}{'Same':<10}{1:<8}{input_size}x{input_size}")
for k in [3, 5, 7]:
    P = 0
    out = output_size(input_size, k, P, 1)
    print(f"{k}x{k:<7}{'Valid':<10}{1:<8}{out}x{out}")


In [ ]:
# Demonstrate with actual Keras Conv2D layers on a batch of real images
sample_batch = X_train[:1].astype('float32') / 255.0

print(f"{'Kernel':<8}{'Padding':<10}{'Output Shape'}")
for k in [3, 5, 7]:
    for pad in ['same', 'valid']:
        conv = layers.Conv2D(8, (k, k), padding=pad)
        out = conv(sample_batch)
        print(f"{k}x{k:<5}{pad:<10}{out.shape}")


**Inference:** With `same` padding the output retains the same spatial resolution (32×32) regardless
of kernel size, since the input is zero-padded to compensate for the larger receptive field. With `valid`
padding the output shrinks more as the kernel grows, since larger kernels cannot be centred as close to
the image border.

## Task 3 — Study Hyperparameters
Compare stride (1 vs 2) and padding (same vs valid) using the formula
$$\left\lfloor\frac{N-F+2P}{S}\right\rfloor + 1$$
for a 32×32 input and a 3×3 kernel.


In [ ]:
print(f"{'Stride':<8}{'Padding':<10}{'P':<4}{'Output Size'}")
configs = [(1, 'same', 1), (1, 'valid', 0), (2, 'same', 1), (2, 'valid', 0)]
for S, pad_name, P in configs:
    out = output_size(32, 3, P, S)
    print(f"{S:<8}{pad_name:<10}{P:<4}{out}x{out}")


In [ ]:
# Verify with Keras layers
sample_batch = X_train[:1].astype('float32') / 255.0
print(f"{'Stride':<8}{'Padding':<10}{'Output Shape'}")
for S in [1, 2]:
    for pad in ['same', 'valid']:
        conv = layers.Conv2D(8, (3, 3), strides=S, padding=pad)
        out = conv(sample_batch)
        print(f"{S:<8}{pad:<10}{out.shape}")


**Inference:** Increasing the stride reduces the output spatial resolution roughly in proportion to
the stride value, while `same` padding keeps the output an exact multiple related to the stride (here
halving it cleanly), whereas `valid` padding discards border pixels that do not fit a complete kernel
window.

## Preprocessing
Normalize pixel values to [0, 1] and one-hot encode the labels before building the CNN used in Tasks 4–7.


In [ ]:
X_train_norm = X_train.astype('float32') / 255.0
X_test_norm = X_test.astype('float32') / 255.0

y_train_cat = keras.utils.to_categorical(y_train, 10)
y_test_cat = keras.utils.to_categorical(y_test, 10)

# Hold out a validation split from the training set
val_split = 0.1
n_val = int(len(X_train_norm) * val_split)
X_val, y_val = X_train_norm[:n_val], y_train_cat[:n_val]
X_tr, y_tr = X_train_norm[n_val:], y_train_cat[n_val:]

print("Train:", X_tr.shape, " Val:", X_val.shape, " Test:", X_test_norm.shape)


## Task 6 — Construct and Train the CNN

Architecture:
`Input → [Conv → BatchNorm → ReLU → MaxPool] ×3 → Flatten → Dense(128, ReLU) → Dropout(0.5) → Dense(10, Softmax)`

Trained with:
- Optimizer: Adam
- Epochs: up to 20 (early stopping, patience = 3 on validation loss)
- Batch size: 64


In [ ]:
def build_cnn(input_shape=(32, 32, 3), num_classes=10):
    model = models.Sequential([
        layers.Input(shape=input_shape),

        layers.Conv2D(32, (3, 3), padding='same', name='conv2d_1'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(64, (3, 3), padding='same', name='conv2d_2'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(128, (3, 3), padding='same', name='conv2d_3'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D((2, 2)),

        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ])
    return model

model = build_cnn()
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])
model.summary()


In [ ]:
# First conv layer parameter check (32 filters of 3x3x3 -> matches Discussion Q6)
first_conv_params = (3*3*3 + 1) * 32
print("Expected first Conv2D params:", first_conv_params)


In [ ]:
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=3, restore_best_weights=True
)

start = time.time()
history = model.fit(
    X_tr, y_tr,
    validation_data=(X_val, y_val),
    epochs=20,
    batch_size=64,
    callbacks=[early_stop],
    verbose=1
)
train_time = time.time() - start
print(f"Training completed in {train_time/60:.2f} minutes")


In [ ]:
# Training vs Validation Accuracy and Loss
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(history.history['accuracy'], label='Training')
axes[0].plot(history.history['val_accuracy'], label='Validation')
axes[0].set_title('Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(history.history['loss'], label='Training')
axes[1].plot(history.history['val_loss'], label='Validation')
axes[1].set_title('Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle('Training vs Validation Accuracy and Loss')
plt.tight_layout()
plt.show()


**Inference:** Training accuracy rises steadily while validation accuracy climbs more slowly and
plateaus, showing that the model continues to fit the training data faster than it generalises. A growing
gap between training and validation loss indicates the onset of mild overfitting, which is why early
stopping (patience = 3, monitoring validation loss) halts training once validation loss stops improving.

## Task 4 — Visualize Feature Maps
Extract and display feature maps from the three convolutional layers (`conv2d_1`, `conv2d_2`, `conv2d_3`)
for a single test image, showing the first 8 channels of each layer's output.


In [ ]:
layer_names = ['conv2d_1', 'conv2d_2', 'conv2d_3']
outputs = [model.get_layer(name).output for name in layer_names]
activation_model = models.Model(inputs=model.inputs, outputs=outputs)

test_img = X_test_norm[0:1]
activations = activation_model.predict(test_img, verbose=0)

fig, axes = plt.subplots(len(layer_names), 8, figsize=(16, 6))
for row, (layer_name, activation) in enumerate(zip(layer_names, activations)):
    for col in range(8):
        ax = axes[row, col]
        ax.imshow(activation[0, :, :, col], cmap='viridis')
        ax.axis('off')
        if col == 0:
            ax.set_ylabel(layer_name, fontsize=9)

plt.suptitle("Feature Maps by Convolutional Layer")
plt.tight_layout()
plt.show()


**Inference:** The first convolutional layer's feature maps highlight low-level structures such as
edges, colour contrasts, and object silhouettes that are still visually recognisable. Deeper layers
(`conv2d_2`, `conv2d_3`) produce increasingly abstract, coarser activation patterns as spatial resolution
is reduced by pooling, reflecting the hierarchical feature extraction typical of CNNs.

## Task 5 — Max Pooling vs Average Pooling
Compare output size and (approximate) effect on accuracy for Max Pooling vs Average Pooling, using a
smaller/faster model trained for a few epochs on a subset for a quick empirical comparison.


In [ ]:
def output_size_pool(N, F=2, S=2):
    return (N - F) // S + 1

print("Max/Avg Pool 2x2 stride 2 on 32x32 input -> output:",
      output_size_pool(32), "x", output_size_pool(32))


In [ ]:
def build_cnn_pooling(pool_type='max'):
    Pool = layers.MaxPooling2D if pool_type == 'max' else layers.AveragePooling2D
    model = models.Sequential([
        layers.Input(shape=(32, 32, 3)),
        layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
        Pool((2, 2)),
        layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
        Pool((2, 2)),
        layers.Flatten(),
        layers.Dense(64, activation='relu'),
        layers.Dense(10, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

# Quick comparison on a subset for speed
subset = 10000
results = {}
for pool_type in ['max', 'avg']:
    m = build_cnn_pooling(pool_type)
    h = m.fit(X_tr[:subset], y_tr[:subset],
              validation_data=(X_val, y_val),
              epochs=5, batch_size=64, verbose=0)
    val_acc = h.history['val_accuracy'][-1]
    results[pool_type] = val_acc
    print(f"{pool_type.capitalize()} Pooling — final validation accuracy: {val_acc:.4f}")

print("\nComparison:", results)


**Inference:** Both Max Pooling and Average Pooling with a 2×2 window and stride 2 reduce the
spatial dimensions of a feature map identically (by half in each direction), since output size depends
only on window size and stride, not on the pooling operation. Max Pooling propagates only the strongest
activation in each window, which tends to preserve sharp edges and salient features, generally yielding
marginally higher accuracy for natural image datasets like CIFAR-10, while Average Pooling smooths
activations and can slightly reduce sensitivity to noise but also dilutes strong discriminative signals.
The full CNN in Task 6 uses Max Pooling throughout.

## Task 7 — Evaluate the Model
Compute Accuracy, Precision, Recall, F1-score, Confusion Matrix, and Classification Report on the test set.


In [ ]:
test_loss, test_acc = model.evaluate(X_test_norm, y_test_cat, verbose=0)
print(f"Test Accuracy: {test_acc*100:.2f}%")
print(f"Test Loss: {test_loss:.4f}")

y_pred_probs = model.predict(X_test_norm, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = y_test

precision = precision_score(y_true, y_pred, average='weighted')
recall = recall_score(y_true, y_pred, average='weighted')
f1 = f1_score(y_true, y_pred, average='weighted')

print(f"\nPrecision (weighted): {precision:.4f}")
print(f"Recall (weighted)   : {recall:.4f}")
print(f"F1-score (weighted) : {f1:.4f}")


In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()


In [ ]:
# Classification Report
print("Classification Report:\n")
print(classification_report(y_true, y_pred, target_names=class_names))


**Inference:** The strong diagonal of the confusion matrix confirms the overall test accuracy. Classes
with more distinctive shapes/colours (e.g. ship, truck, automobile) tend to be classified most reliably,
while visually similar animal classes (e.g. cat/dog, bird/deer) are more frequently confused with one
another at this resolution.

## 11. Results Summary

In [ ]:
print("="*50)
print("RESULTS SUMMARY")
print("="*50)
print(f"Training Accuracy (final epoch)   : {history.history['accuracy'][-1]*100:.2f}%")
print(f"Validation Accuracy (final epoch) : {history.history['val_accuracy'][-1]*100:.2f}%")
print(f"Testing Accuracy                  : {test_acc*100:.2f}%")
print(f"Precision (weighted)              : {precision:.4f}")
print(f"Recall (weighted)                 : {recall:.4f}")
print(f"F1-score (weighted)               : {f1:.4f}")

total_params = model.count_params()
trainable_params = sum(np.prod(v.shape) for v in model.trainable_weights)
non_trainable_params = total_params - trainable_params
print(f"Total Parameters                  : {total_params}")
print(f"Trainable Parameters              : {int(trainable_params)}")
print(f"Non-trainable Parameters          : {int(non_trainable_params)}")


## 12. Discussion

**1. Why is convolution preferred over fully connected layers for images?**
Convolution exploits spatial structure through local connectivity and parameter sharing: each filter
slides across the entire image using the same small set of weights, so it can detect a feature (e.g. an
edge) regardless of where it appears. A fully connected layer connects every input pixel to every
neuron, ignoring spatial locality and requiring a separate weight per pixel–neuron pair — far less
parameter-efficient and translation-invariant.

**2. How does stride affect the feature map size?**
Stride is the step size by which the kernel moves; a larger stride skips more pixels between
applications, producing a smaller output feature map. This is captured by
$\frac{N-F+2P}{S}+1$ — doubling the stride roughly halves the output width and height.

**3. What is the role of padding?**
Padding adds extra (typically zero-valued) border pixels before convolution. It (1) controls output
size — `same` padding preserves spatial dimensions, `valid` shrinks them — and (2) ensures border pixels
are convolved as many times as central pixels, preventing their information from being under-represented.

**4. Why is pooling used?**
Pooling down-samples feature maps, reducing spatial dimensions and downstream parameter count, lowering
computational cost and helping control overfitting. It also introduces translation invariance, since
small shifts within a pooling window do not change the pooled output.

**5. How do feature maps represent image characteristics?**
Each feature map is the output of a specific learned filter, and its activation pattern shows where a
particular pattern (edge, colour blob, texture, or more abstract shape in deeper layers) is present.
Early-layer maps resemble edge/contrast-detected versions of the image; deeper-layer maps become
increasingly abstract, encoding higher-level semantic concepts.

**6. Why do CNNs require fewer parameters than MLPs?**
CNNs use parameter sharing (the same small kernel reused across all spatial locations) and local
receptive fields, so a convolutional layer's parameter count depends only on kernel size and number of
filters — not on input image size. An MLP's fully connected layer requires weights scaling with the
product of input size and hidden layer size, which becomes far larger even for modest images.


## 10. Additional Exercises

In [ ]:
# 1. Output size for a 64x64 image, 5x5 kernel, stride 2, padding 2
N, F, P, S = 64, 5, 2, 2
out = (N - F + 2*P)//S + 1
print(f"1. Output size (64x64, 5x5 kernel, stride 2, padding 2): {out}x{out}")

# 2. Trainable parameters for a conv layer with 64 filters of size 3x3, RGB input
params_ex2 = (3*3*3 + 1) * 64
print(f"2. Trainable parameters (64 filters, 3x3, RGB input): {params_ex2}")


In [ ]:
# 3. Compare ReLU vs Sigmoid activation functions
x = np.linspace(-10, 10, 200)
relu = np.maximum(0, x)
sigmoid = 1 / (1 + np.exp(-x))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(x, relu)
axes[0].set_title("ReLU")
axes[0].grid(alpha=0.3)
axes[1].plot(x, sigmoid)
axes[1].set_title("Sigmoid")
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("ReLU: f(x)=max(0,x). Computationally cheap, avoids vanishing gradients for x>0,\n"
      "but can suffer from 'dying ReLU' for x<0. Widely used in hidden layers of CNNs.\n\n"
      "Sigmoid: f(x)=1/(1+e^-x). Squashes output to (0,1), useful for binary probabilities,\n"
      "but saturates for large |x| causing vanishing gradients, and is more expensive to compute.\n"
      "ReLU is generally preferred in deep CNN hidden layers for faster, more stable training.")


**4. Replace Max Pooling with Average Pooling and compare results** — see Task 5 above, where both
pooling types were trained and compared empirically.

**5. Increase filters from 16 to 64 and analyze effect on accuracy/computation time** — increasing the
number of filters increases the model's representational capacity (more learnable feature detectors per
layer) and its parameter count/FLOPs, typically improving accuracy up to a point at the cost of longer
training and inference time; beyond a certain width the accuracy gains diminish while compute cost keeps
rising (diminishing returns / risk of overfitting on a fixed-size dataset like CIFAR-10).

In [ ]:
# Quick empirical check: 16 vs 64 filters in the first conv layer
def build_cnn_filters(n_filters):
    model = models.Sequential([
        layers.Input(shape=(32, 32, 3)),
        layers.Conv2D(n_filters, (3, 3), padding='same', activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(n_filters*2, (3, 3), padding='same', activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(64, activation='relu'),
        layers.Dense(10, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

subset = 10000
for n_filters in [16, 64]:
    m = build_cnn_filters(n_filters)
    t0 = time.time()
    h = m.fit(X_tr[:subset], y_tr[:subset],
              validation_data=(X_val, y_val),
              epochs=5, batch_size=64, verbose=0)
    elapsed = time.time() - t0
    print(f"Filters={n_filters:<4} Params={m.count_params():<10} "
          f"Val Acc={h.history['val_accuracy'][-1]:.4f}  Time={elapsed:.1f}s")


## Save the Trained Model

In [ ]:
model.save('cnn_cifar10_model.keras')
print("Model saved as cnn_cifar10_model.keras")
